# Kriteria 1 — Fine-tuning LLM dengan QLoRA (Unsloth + SFTTrainer)

**Submission PGABL — Fepriyadi Harahap**

Notebook ini melatih model *text generation* menjadi asisten instruksi berbahasa Indonesia.
Hasilnya di-*push* ke Hugging Face dan dipakai lagi pada tahap GRPO dan RAG.

| Komponen | Pilihan |
|---|---|
| Pre-trained model | `unsloth/Qwen2.5-3B-Instruct` (Qwen2, task Text Generation, didukung Unsloth) |
| Dataset | `Ichsan2895/alpaca-gpt4-indonesian` (instruction tanya–jawab format Alpaca, Bahasa Indonesia) |
| Chat template | ChatML (`<|im_start|>` / `<|im_end|>`) |
| Teknik | QLoRA: 4-bit NF4 + **double quantization**, LoRA di Multi-Head Attention **dan** Feed Forward Network |
| Trainer | `SFTTrainer` (TRL) — eksperimen hyperparameter, lalu training final **800 steps** dengan validation |
| Output | `model.push_to_hub_merged(..., save_method="merged_16bit")` |

> Jalankan di **Google Colab (GPU T4)** atau **Kaggle (GPU T4/P100)**. Simpan token Hugging Face (akses *write*)
> sebagai secret bernama `HF_TOKEN`.

## 0. Instalasi & Import

In [ ]:
%%capture
# Unsloth otomatis memasang versi torch / transformers / trl / peft / bitsandbytes yang saling kompatibel.
# Jika Colab meminta "Restart session" setelah instalasi, lakukan restart lalu lanjutkan dari cell berikutnya.
!pip install -q unsloth

In [ ]:
# Unsloth harus di-import sebelum transformers / trl agar patch optimasinya aktif.
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only

import gc
import os
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, BitsAndBytesConfig, TextStreamer
from trl import SFTConfig, SFTTrainer

In [ ]:
assert torch.cuda.is_available(), "Notebook ini membutuhkan GPU CUDA (Colab T4 / Kaggle T4 / P100)."
gpu = torch.cuda.get_device_properties(0)
GPU_TOTAL_GB = round(gpu.total_memory / 1024**3, 2)
print(f"GPU: {gpu.name} | VRAM: {GPU_TOTAL_GB} GB | bf16 didukung: {is_bfloat16_supported()}")

In [ ]:
from huggingface_hub import login, whoami


def get_hf_token():
    """Token Hugging Face (akses write) dari Colab Secrets / Kaggle Secrets / env var / input manual."""
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    from getpass import getpass
    return getpass("Masukkan Hugging Face token (write): ")


HF_TOKEN = get_hf_token()
login(token=HF_TOKEN)
HF_USERNAME = whoami(token=HF_TOKEN)["name"]
print("Login Hugging Face sebagai:", HF_USERNAME)

In [ ]:
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"   # arsitektur Qwen2 (didukung Unsloth), task: Text Generation
SFT_REPO = f"{HF_USERNAME}/Qwen2.5-3B-Instruct-Alpaca-ID-SFT"     # hasil Kriteria 1 (SFT)
GRPO_REPO = f"{HF_USERNAME}/Qwen2.5-3B-Instruct-Alpaca-ID-GRPO"   # hasil Kriteria 1 Advanced (GRPO)
SEED = 3407

MAX_SEQ_LENGTH = 1024
SFT_SYSTEM_PROMPT = "Anda adalah asisten AI berbahasa Indonesia yang membantu, jujur, dan menjawab dengan jelas."
print("Base model :", BASE_MODEL)
print("Repo tujuan:", SFT_REPO)

## 1. Dataset & Mapping ke Chat Template (ChatML)

Dataset `Ichsan2895/alpaca-gpt4-indonesian` **tidak** memakai tiga kolom Alpaca standar
(`instruction`, `input`, `output`). Isinya hanya:

```text
Unnamed: 0   -> nomor baris asli (tidak dipakai)
input        -> instruksi pada baris pertama + konteks opsional pada baris berikutnya
output       -> jawaban (ground truth)
```

Karena itu fungsi mapping memecah `input` menjadi *instruksi* dan *konteks* terlebih dahulu,
lalu menyusun percakapan `system → user → assistant` dan merendernya dengan chat template ChatML
milik Qwen2.5 melalui `datasets.Dataset.map`.

In [ ]:
raw_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print(raw_dataset)
n_with_context = sum("\n" in text.strip() for text in raw_dataset["input"])
print(f"Baris dengan konteks tambahan: {n_with_context} dari {len(raw_dataset)}")

### Contoh satu baris dataset yang **BELUM** di-mapping

In [ ]:
raw_dataset[2]

In [ ]:
# Tokenizer + chat template ChatML (template "qwen-2.5" di Unsloth = ChatML dengan <|im_start|> / <|im_end|>)
tokenizer = get_chat_template(AutoTokenizer.from_pretrained(BASE_MODEL), chat_template="qwen-2.5")


def split_alpaca_input(text):
    """Kolom `input` = instruksi (baris pertama) + konteks opsional (baris berikutnya)."""
    instruction, _, context = text.strip().partition("\n")
    return instruction.strip(), context.strip()


def to_chatml(batch):
    """Mapping batch Alpaca mentah -> teks percakapan ChatML lengkap dengan token spesial."""
    texts = []
    for raw_input, output in zip(batch["input"], batch["output"]):
        instruction, context = split_alpaca_input(raw_input)
        user_content = f"{instruction}\n\nInput:\n{context}" if context else instruction
        messages = [
            {"role": "system", "content": SFT_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output.strip()},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return {"text": texts}


dataset = raw_dataset.map(to_chatml, batched=True, remove_columns=raw_dataset.column_names)

### Contoh satu baris dataset yang **SUDAH** di-mapping (lengkap dengan token spesial)

In [ ]:
print(dataset[2]["text"])

In [ ]:
# Buang sampel yang lebih panjang dari MAX_SEQ_LENGTH agar tidak terpotong saat training.
dataset = dataset.map(
    lambda batch: {"n_tokens": [len(ids) for ids in tokenizer(batch["text"]).input_ids]},
    batched=True,
)
before = len(dataset)
dataset = dataset.filter(lambda row: row["n_tokens"] <= MAX_SEQ_LENGTH).remove_columns("n_tokens")
print(f"Sampel <= {MAX_SEQ_LENGTH} token: {len(dataset)} dari {before}")

### Split train / validation

Validation set dipakai untuk `eval_dataset` di trainer, sehingga kurva *eval loss* bisa dibandingkan
dengan *train loss* untuk mendeteksi overfitting.

In [ ]:
split = dataset.train_test_split(test_size=500, seed=SEED)
train_dataset, eval_dataset = split["train"], split["test"]
print("Train     :", len(train_dataset))
print("Validation:", len(eval_dataset))

## 2. QLoRA: Model 4-bit dengan Double Quantization + Adapter LoRA

- `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)`
  → bobot model dibekukan dalam 4-bit NF4; *double quantization* ikut mengkuantisasi konstanta kuantisasinya
  sehingga VRAM lebih hemat.
- `use_exact_model_name=True` memastikan Unsloth memuat checkpoint asli lalu mengkuantisasinya **dengan
  konfigurasi di atas** (bukan checkpoint pra-kuantisasi).
- Adapter LoRA dipasang pada **kedua** komponen komputasi utama tiap blok decoder:

```text
Qwen2DecoderLayer
    ├── self_attn (Multi-Head Attention) ── q_proj, k_proj, v_proj, o_proj   <- LoRA
    └── mlp (Feed Forward Network) ──────── gate_proj, up_proj, down_proj    <- LoRA
```

In [ ]:
COMPUTE_DTYPE = torch.bfloat16 if is_bfloat16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,   # double quantization
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

ATTENTION_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]   # Multi-Head Attention
FFN_MODULES = ["gate_proj", "up_proj", "down_proj"]            # Feed Forward Network (SwiGLU MLP)


def load_qlora_model(lora_r, lora_alpha):
    """Muat base model dalam 4-bit (QLoRA) lalu pasang adapter LoRA. Hanya adapter yang dilatih."""
    model, tok = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        quantization_config=bnb_config,
        use_exact_model_name=True,
    )
    tok = get_chat_template(tok, chat_template="qwen-2.5")
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.0,
        target_modules=ATTENTION_MODULES + FFN_MODULES,
        bias="none",
        use_gradient_checkpointing="unsloth",   # hemat VRAM -> mencegah OOM
        random_state=SEED,
    )
    return model, tok


def free_gpu():
    """Kosongkan cache VRAM. Hapus dulu referensinya di pemanggil (`del model`) sebelum memanggil ini."""
    gc.collect()
    torch.cuda.empty_cache()

### Verifikasi konfigurasi QLoRA

Model dimuat sekali di sini untuk menunjukkan bahwa kuantisasi 4-bit + double quant aktif dan LoRA
terpasang di modul attention dan FFN.

In [ ]:
model, tokenizer = load_qlora_model(lora_r=16, lora_alpha=32)

qcfg = model.config.quantization_config
qcfg = qcfg if isinstance(qcfg, dict) else qcfg.to_dict()
print({k: qcfg[k] for k in ["load_in_4bit", "bnb_4bit_quant_type", "bnb_4bit_use_double_quant", "bnb_4bit_compute_dtype"]})

model.print_trainable_parameters()
lora_targets = sorted({name.split(".")[-2] for name, _ in model.named_modules() if name.endswith("lora_A")})
print("Modul ber-LoRA:", lora_targets)
print("Contoh layer:", model.base_model.model.model.layers[0].self_attn.q_proj)

In [ ]:
del model
free_gpu()

## 3. Eksperimen Hyperparameter (Train + Validation)

Dua kombinasi hyperparameter dibandingkan dengan budget yang sama (200 steps, evaluasi tiap 50 steps).
Konfigurasi trainer memakai `eval_dataset`, `eval_strategy="steps"` (nama baru dari
`evaluation_strategy` di transformers ≥ 4.46) dan logging tiap 10 steps.

| Eksperimen | LoRA r / alpha | Learning rate | Scheduler | Hipotesis |
|---|---|---|---|---|
| `exp_A` | 16 / 32 | 2e-4 | linear | konvergen cepat, risiko overfit lebih tinggi |
| `exp_B` | 32 / 32 | 5e-5 | cosine | kapasitas adapter lebih besar, update lebih halus |

In [ ]:
EXPERIMENTS = {
    "exp_A": dict(lora_r=16, lora_alpha=32, learning_rate=2e-4, lr_scheduler_type="linear"),
    "exp_B": dict(lora_r=32, lora_alpha=32, learning_rate=5e-5, lr_scheduler_type="cosine"),
}
EXPERIMENT_STEPS = 200
FINAL_STEPS = 800


def make_trainer(model, tok, run_name, cfg, max_steps, eval_steps):
    args = SFTConfig(
        output_dir=f"outputs/{run_name}",
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,          # effective batch = 8
        per_device_eval_batch_size=4,
        max_steps=max_steps,
        learning_rate=cfg["learning_rate"],
        lr_scheduler_type=cfg["lr_scheduler_type"],
        warmup_steps=max(10, max_steps // 20),
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        eval_strategy="steps",                  # evaluation_strategy = "steps"
        eval_steps=eval_steps,
        logging_strategy="steps",
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        dataset_num_proc=2,
    )
    trainer = SFTTrainer(
        model=model,
        processing_class=tok,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=args,
    )
    # Loss hanya dihitung pada jawaban assistant (prompt system/user di-mask).
    return train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )


def loss_history(trainer):
    logs = pd.DataFrame(trainer.state.log_history)
    train = logs.dropna(subset=["loss"])[["step", "loss"]] if "loss" in logs else pd.DataFrame()
    evals = logs.dropna(subset=["eval_loss"])[["step", "eval_loss"]] if "eval_loss" in logs else pd.DataFrame()
    return train, evals

In [ ]:
experiment_results = {}

for run_name, cfg in EXPERIMENTS.items():
    print(f"\n===== {run_name}: {cfg} =====")
    model, tok = load_qlora_model(cfg["lora_r"], cfg["lora_alpha"])
    trainer = make_trainer(model, tok, run_name, cfg, max_steps=EXPERIMENT_STEPS, eval_steps=50)
    start = time.time()
    trainer.train()
    train_log, eval_log = loss_history(trainer)
    experiment_results[run_name] = dict(cfg=cfg, train=train_log, eval=eval_log, minutes=(time.time() - start) / 60)
    del trainer, model, tok
    free_gpu()

In [ ]:
fig, axes = plt.subplots(1, len(experiment_results), figsize=(13, 4), sharey=True)
for ax, (run_name, res) in zip(axes, experiment_results.items()):
    ax.plot(res["train"]["step"], res["train"]["loss"], label="train loss", alpha=0.8)
    ax.plot(res["eval"]["step"], res["eval"]["eval_loss"], "o-", label="eval loss")
    ax.set_title(f"{run_name}  (r={res['cfg']['lora_r']}, lr={res['cfg']['learning_rate']})")
    ax.set_xlabel("step")
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("loss")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for run_name, res in experiment_results.items():
    eval_losses = res["eval"]["eval_loss"].tolist()
    final_train = res["train"]["loss"].tail(5).mean()
    rows.append({
        "experiment": run_name,
        **res["cfg"],
        "train_loss (avg 5 log terakhir)": round(final_train, 4),
        "eval_loss akhir": round(eval_losses[-1], 4),
        "eval_loss terbaik": round(min(eval_losses), 4),
        "gap eval - train": round(eval_losses[-1] - final_train, 4),
        # overfitting: eval loss naik lagi > 2% dari titik terendahnya
        "overfitting": eval_losses[-1] > min(eval_losses) * 1.02,
        "durasi (menit)": round(res["minutes"], 1),
    })
summary = pd.DataFrame(rows).set_index("experiment")
display(summary)

candidates = summary[~summary["overfitting"]] if (~summary["overfitting"]).any() else summary
BEST_EXPERIMENT = candidates["eval_loss akhir"].idxmin()
BEST_CONFIG = EXPERIMENTS[BEST_EXPERIMENT]
print(f"Kombinasi terbaik: {BEST_EXPERIMENT} -> {BEST_CONFIG}")

## 4. Training Final — 800 Steps dengan Kombinasi Terbaik

Kombinasi hyperparameter dengan *eval loss* akhir terendah (dan tanpa tanda overfitting) dilatih ulang
dari base model selama **800 steps** dengan evaluasi tiap 100 steps.

In [ ]:
model, tokenizer = load_qlora_model(BEST_CONFIG["lora_r"], BEST_CONFIG["lora_alpha"])
trainer = make_trainer(model, tokenizer, "final_sft", BEST_CONFIG, max_steps=FINAL_STEPS, eval_steps=100)

torch.cuda.reset_peak_memory_stats()
trainer_stats = trainer.train()

In [ ]:
peak_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"Steps selesai      : {trainer.state.global_step} / {FINAL_STEPS}")
print(f"Durasi training    : {trainer_stats.metrics['train_runtime'] / 60:.1f} menit")
print(f"Train loss (rata2) : {trainer_stats.metrics['train_loss']:.4f}")
print(f"Puncak VRAM        : {peak_gb} GB dari {GPU_TOTAL_GB} GB (tanpa OOM)")
assert trainer.state.global_step >= 800

final_train_log, final_eval_log = loss_history(trainer)
plt.figure(figsize=(8, 4))
plt.plot(final_train_log["step"], final_train_log["loss"], label="train loss", alpha=0.8)
plt.plot(final_eval_log["step"], final_eval_log["eval_loss"], "o-", label="eval loss")
plt.title(f"Training final ({BEST_EXPERIMENT}, {FINAL_STEPS} steps)")
plt.xlabel("step")
plt.ylabel("loss")
plt.grid(alpha=0.3)
plt.legend()
plt.show()
display(final_eval_log.set_index("step").T)

## 5. Uji Inferensi Model Hasil Fine-tuning

In [ ]:
FastLanguageModel.for_inference(model)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

test_questions = [
    "Jelaskan perbedaan antara pekerja kontrak (PKWT) dan pekerja tetap (PKWTT) secara singkat.",
    "Berikan tiga tips menjaga kesehatan mental bagi pekerja kantoran.",
]
for question in test_questions:
    print(f"\n### {question}")
    messages = [{"role": "system", "content": SFT_SYSTEM_PROMPT}, {"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    _ = model.generate(**inputs, streamer=streamer, max_new_tokens=256, temperature=0.3, top_p=0.9, do_sample=True)

## 6. Push Model ke Hugging Face (`merged_16bit`)

Adapter LoRA digabung ke bobot base model lalu diunggah dalam presisi 16-bit, sehingga dapat dimuat
langsung pada tahap GRPO dan RAG.

In [ ]:
def save_link(label, url, path="link_huggingface.txt"):
    """Catat tautan model di link_huggingface.txt; baris berlabel sama diganti (aman di-run ulang)."""
    lines = open(path).read().splitlines() if os.path.exists(path) else []
    lines = [line for line in lines if line.strip() and not line.startswith(label)] + [f"{label}: {url}"]
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(open(path).read())


model.push_to_hub_merged(SFT_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
SFT_URL = f"https://huggingface.co/{SFT_REPO}"
print("Model SFT tersedia di:", SFT_URL)

save_link("Model Fine-tuning (SFT, Kriteria 1)", SFT_URL)

## Kesimpulan

- Dataset Alpaca Indonesia di-*mapping* ke ChatML dengan `Dataset.map`, contoh baris sebelum/sesudah ditampilkan.
- Model dimuat dengan QLoRA (4-bit NF4 + double quantization); LoRA terpasang di Multi-Head Attention
  dan Feed Forward Network.
- Dua eksperimen hyperparameter dibandingkan lewat kurva train/eval loss; kombinasi terbaik dilatih
  800 steps tanpa OOM.
- Model hasil merge 16-bit di-*push* ke Hugging Face dan tautannya dicatat di `link_huggingface.txt`.

**Langkah berikutnya:** notebook `GRPO_submission_PGABL_FepriyadiH.ipynb` memuat `SFT_REPO` dan melatih penalaran `<think>`.